# Machine Learning Homework - Group 22
# Notebook 1 - Import & Explore

**Objective**: Inspect `train.csv` and the auxiliary `clientes.csv`, describe the
observations and variables, provide descriptive statistics, check data quality and
inconsistencies and explore univariate and multivariate relationships.

**Contents**:

1. Environment
2. Loading the files
3. Structural analysis of train dataset
4. Column roles
5. Readable labels for columns
6. Duplicates and uniqueness of identifiers
7. Target and baseline to beat
8. Missing data
9. Invalid values
10. Statistical analysis of numeric features
11. Histograms and outlier fences
12. Statistical analysis for categorical features
13. Correlation between numeric features and the target
14. Correlation among numeric features
15. Correlation between categorical features and the target
16. Customer metadata
17. Train vs test
18. Summary

No analysis here drops features or rows: every table and figure covers all
29 numeric and all 6 categorical features.

### Cell 1 - Environment:
This is where the import takes place, the paths are fixed and a visual theme is chosen for all the figures.

`Path.cwd()` is the only path helper used, because notebooks run from inside `01_notebooks/`.
The `data/` path printed at the bottom confirms the folder layout.
A `guide` function is created to append a "how to read" caption to the axes title, so a figure still explains itself
when it is printed on a page or opened on its own.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy.stats import chi2_contingency, pointbiserialr
from scipy.stats.contingency import association
pd.set_option('display.max_columns', None)


ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
FIG_DIR = ARTIFACTS_DIR / "figures"

PLOT_BLUE = "#1779A8"
PLOT_ORANGE = "#C96A00"

def guide(ax, text):
    ax.set_title(ax.get_title() + "\n" + text, fontsize=9)

# seaborn 0.13 still calls ax.boxplot(vert=...), deprecated in matplotlib 3.11;
# the warning is raised inside seaborn, not by anything these cells pass in
warnings.filterwarnings("ignore", message="vert: bool was deprecated")

sns.set_theme(style="whitegrid")
print("Environment ready.")
print(" Repo root:", ROOT)
print(" Data dir :", DATA_DIR)

### Cell 2 - Loading the files:
Read the raw inputs and their dimensions and confirm the target column exists where it should.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
clientes = pd.read_csv(DATA_DIR / "clientes.csv")
TARGET = "repeat_purchase_90d"

print("Dimensions (rows, columns):")
print("  train        :", train.shape)
print("  test         :", test.shape)
print("  clientes     :", clientes.shape)

print("\nTarget info:")
print("  target in train :", TARGET in train.columns)
print("  target in test  :", TARGET in test.columns)
print("  target values   :", sorted(train[TARGET].unique().tolist()))

### Cell 3 - Structural analysis of train dataset:
See what is actually in the data before classifying anything.

In [ ]:
print("Train columns and data types:")
print(train.dtypes.to_string())
print("\nFirst 5 rows of train:")
train.head()

### Cell 4 - Column roles:
Decide how each column will be treated: identifier, date, numeric or categorical (target already identified in Cell 2).

From the previous cell, it can be observed that `purchase_date` is stored as string instead of datetime. Let's also compare the `train` and `test` columns once the target is removed, to make sure there aren't any exclusive ones.

In [ ]:
id_cols = ["ID", "customer_id"]
date_cols = ["purchase_date"]

numeric_all = train.select_dtypes("number").columns.tolist()
categorical_cols = [c for c in train.select_dtypes(include=["object", "str"]).columns if c not in date_cols]
numeric_cols = [c for c in numeric_all if c != TARGET and c not in id_cols]

print("\nColumns by role in train:")
print(f"identifiers        : {len(id_cols)}, {id_cols}")
print(f"target             : {TARGET}")
print(f"date (as string)   : {date_cols}")
print(f"numeric            : {len(numeric_cols)}, {numeric_cols}")
print(f"categorical        : {len(categorical_cols)}, {categorical_cols}")

only_train = sorted(set(train.columns) - set(test.columns) - {TARGET})
only_test = sorted(set(test.columns) - set(train.columns))
print("\nSchema alignment:")
print("columns only in train (besides target):", only_train if only_train else "none")
print("columns only in test                  :", only_test if only_test else "none")

### Cell 5 - Readable labels for columns:
Map each raw column name to a label used in all tables and figures in this notebook using a dictionary (`FEATURE_LABELS`).

`clean_label` is a function that converts column names into more readable labels. It first checks for a custom label in the dictionary. If there isn't one, it replaces underscores with spaces and uppercases the first letter of each word. `train columns without an explicit label: none` proves that there are no columns left to label.

In [ ]:
FEATURE_LABELS = {
    "Total a Pagar": "Total Payable (EUR)",
    "Total Bruto": "Gross Total (EUR)",
    "Total Liquido": "Net Total (EUR)",
    "Total Imp_": "Tax Total (EUR)",
    "Total Portes": "Shipping Cost (EUR)",
    "Total Desc_ Global": "Global Discount (EUR)",
    "Total Desc_ Linha": "Line Discount (EUR)",
    "Custo": "Cost of Goods (EUR)",
    "N_ Detalhes": "Number of Line Items",
    "invoice_count_current_purchase": "Invoice Count in Purchase",
    "repeat_purchase_90d": "Repeat Purchase within 90 Days",
    "days_since_previous_purchase": "Days Since Previous Purchase",
    "customer_tenure_days": "Customer Tenure (Days)",
    "prior_purchase_count": "Prior Purchase Count",
    "prior_total_spend": "Prior Total Spend (EUR)",
    "prior_mean_purchase_value": "Prior Mean Purchase Value (EUR)",
    "prior_std_purchase_value": "Prior Std Purchase Value (EUR)",
    "prior_min_purchase_value": "Prior Min Purchase Value (EUR)",
    "prior_max_purchase_value": "Prior Max Purchase Value (EUR)",
    "prior_mean_gap_days": "Prior Mean Purchase Gap (Days)",
    "prior_std_gap_days": "Prior Std Purchase Gap (Days)",
    "purchase_count_30d": "Purchase Count (Past 30d)",
    "spend_30d": "Spend (Past 30d, EUR)",
    "purchase_count_90d": "Purchase Count (Past 90d)",
    "spend_90d": "Spend (Past 90d, EUR)",
    "purchase_count_180d": "Purchase Count (Past 180d)",
    "spend_180d": "Spend (Past 180d, EUR)",
    "purchase_count_365d": "Purchase Count (Past 365d)",
    "spend_365d": "Spend (Past 365d, EUR)",
    "random_noise": "Random Noise",
    "customer_id": "Customer ID",
    "customer_since_year": "Customer Since (Year)",
    "payment_type": "Payment Method",
    "transport_method": "Transport Method",
    "commercial_zone": "Commercial Zone",
    "salesperson": "Salesperson Code",
    "warehouse": "Warehouse Location",
    "document_series": "Document Series",
    "purchase_date": "Purchase Date",
    "commercial_region": "Commercial Region",
    "country_or_region": "Country / Region",
}


def clean_label(col):
    return FEATURE_LABELS.get(col, col.replace("_", " ").title())


uncovered = [c for c in train.columns if c not in FEATURE_LABELS and c != "ID"]
print("Labels defined for", len(FEATURE_LABELS), "columns.")
print("train columns without an explicit label:", uncovered if uncovered else "none (ID excluded)")

### Cell 6 - Duplicates and uniqueness of identifiers:
Check for repeated id's and duplicated rows within and between `train` and `test`.

Checking duplicates within `train` dataset is important because repeated observations may bias the model by giving certain patterns more significance, leading to overfitting. Identifying identical rows between the `train` and `test` datasets is essential to prevent data leakage, which could lead to an overestimation of the model's performance. In addition, a brief statistical analysis of the number of registered customers and their transactions is performed on the `train` dataset (mean, median, maximum and minimium).

In [ ]:
print("Duplicates/uniqueness:")
print(" train exact duplicate rows:", int(train.duplicated().sum()))
print(" test exact duplicate rows :", int(test.duplicated().sum()))
print(" clientes duplicate rows   :", int(clientes.duplicated().sum()))
print(" train ID unique            :", bool(train["ID"].nunique() == len(train)))
print(" test ID unique             :", bool(test["ID"].nunique() == len(test)))
print(" clientes customer_id unique:", bool(clientes["customer_id"].nunique() == len(clientes)))


measured = [c for c in test.columns if c in train.columns and c != "ID"]
print("\nDuplicate transactions (ID and target excluded):")
print(" train rows involved in duplications:", train.duplicated(subset=measured, keep=False).sum())
print(" train duplicates (surplus)         :", train.duplicated(subset=measured).sum())
print(" test rows involved in duplications :", test.duplicated(subset=measured, keep=False).sum())
print(" test duplicates (surplus)          :", test.duplicated(subset=measured).sum())
train_keys = pd.MultiIndex.from_frame(train[measured])
test_keys = pd.MultiIndex.from_frame(test[measured])
print(" test rows identical to any training row:", test_keys.isin(train_keys).sum())
print("-> no test transaction is a copy of a training one, so the two files are really separate sales")

per_customer = train["customer_id"].value_counts()
print("\nUnique customers:", len(per_customer))
print(f"Transactions per customer (train): mean={round(per_customer.mean(), 1)}, median={per_customer.median()}, max={per_customer.max()}, min={per_customer.min()}")

print("\nCustomers repeat across rows, so a purely random split can place the same customer in both train and validation folds.")
print("\nIn Notebook 4, there will be a grouped or temporal check.")
print("\nTrain rows involved in duplications:")
display(train[train.duplicated(subset=measured, keep=False)])

Looking at the rows involved in duplications, we assume that all those rows result from errors in the dataset that we are unaware of, since there are no features that provide information about the transactions (except for the IDs and the target).
From the transactions per customer analysis, the difference between the mean and median suggests a right-skewed distribution over a wide range, from 1 to 277 transactions per customer.

### Cell 7 - Target and the baseline to beat:
Analyse the distribution of target classes by calculating counts, shares, the majority-class baseline and majority/minority ratio, illustrated by a bar chart.

In [ ]:
counts = train[TARGET].value_counts().sort_index()
share = train[TARGET].value_counts(normalize=True).sort_index()
baseline = share.max()
balance_ratio = share.min() / baseline

print("Target:", TARGET)
target_summary = pd.DataFrame({"Count": counts.values, "Share (%)": (share.values * 100).round(2),
                               "Label": ["repeat buyer" if cls == 1 else "no repeat" for cls in counts.index]}, index=counts.index)
target_summary.index.name = "Class"
display(target_summary)

print(f"\nMajority-class baseline accuracy: {baseline*100:.2f}%")
print(f"Minority/majority ratio: {balance_ratio:.2f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([str(c) for c in counts.index], counts.values, color=[PLOT_BLUE, PLOT_ORANGE], alpha=0.9)

ax.set_ylim(0, counts.values.max() * 1.20)
ax.set_title("Target class balance")
ax.set_xlabel("Class (0 = no repeat, 1 = repeat within 90 days)")
ax.set_ylabel("Transactions")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{share.values[i] * 100:.2f}%", ha="center", fontsize=12, fontweight="bold")
guide(ax, "How to read: counts per class; % above each bar; the taller bar is the accuracy to beat.")
plt.savefig(FIG_DIR / "nb01_01_target_balance.png", dpi=200)
plt.tight_layout()
plt.show()

Looking at the results above, the target distribution is relatively balanced, with a ratio of 0.79, meaning that for every 100 customers who make a new purchase within 90 days, 79 do not. Also, the majority-class baseline accuracy is 55.92%, so any model we develop should achieve a higher accuracy than this.

### Cell 8 - Missing data:
Understand the missing values in our datasets, across columns in `train`, `test` and `clientes` datasets, as well as the number of missing values per row in the `train` dataset.

We also search for a pattern between recency missing values and customers with no previous purchases.

In [ ]:
miss = pd.DataFrame({"train count": train.isna().sum(), "train rows %": train.isna().sum()/len(train)*100,
                     "test count": test.isna().sum(), "test rows %": test.isna().sum()/len(test)*100})
miss = miss.sort_values("train rows %", ascending=False)
miss.insert(0, "label", [clean_label(c) for c in miss.index])
print("Train columns missingness:")
display(miss.round(2))

row_missing = train.isna().sum(axis=1)
print("\nTrain rows missingness:")
display(row_missing.value_counts().sort_index().to_frame())

print("\nClientes columns missingness:")
clientes_miss = pd.DataFrame({"count": clientes.isna().sum(), "rows %": (clientes.isna().sum()/len(clientes)*100).round(2)})
clientes_miss = clientes_miss[(clientes_miss["count"] > 0)].sort_values("count", ascending=False)
clientes_miss.insert(0, "label", [clean_label(c) for c in clientes_miss.index])
display(clientes_miss)

both = int(((train["days_since_previous_purchase"].isna()) & (train["prior_purchase_count"] == 0)).sum())
print("\nRecency and history pattern:")
print("  rows where recency is missing AND prior_purchase_count == 0:", both)

The missing values analysis reveals that historical purchase-related features have the highest proportions of missing values, particularly `prior_std_gap_days` (22.62% of observations) and `prior_mean_gap_days` (20.02% of observations). A interesting observation is that the features with the most missing values are statistical measures (mean, standard deviation, maximum, minimum) rather than values or information about a transaction itself. While 3929 training observations have no missing values at all, more than 1000 already have one missing value, with 36 being the maximum number of missing values in a row (65 observations), suggesting a distinct missingness pattern. Furthermore, 603 observations have missing recency values and no previous purchases, indicating that missing recency values (`days_since_previous_purchase`) may be related to customers with no previous purchases (`prior_purchase_count`).

In clients dataset, `customer_type` is entirely missing and several geographic features have more than half of their values missing (`street`, `island` and `city`).

### Cell 9 - Invalid values:
Find values that are present but cannot be true (negative or extremely high values).

In [ ]:
print("Columns with extremely high values (placeholders):")
extremely_high = pd.DataFrame({"Feature": [clean_label(c) for c in numeric_cols], "Train count": [(train[c]>1e8).sum() for c in numeric_cols],
                               "Test count": [(test[c]>1e8).sum() for c in numeric_cols], "Train max": [train[c].max() for c in numeric_cols],
                               "Test max": [test[c].max() for c in numeric_cols]}, index=numeric_cols)
extremely_high = extremely_high[(extremely_high["Train count"]>0) | (extremely_high["Test count"]>0)]
display(extremely_high)

print("\nColumns with negative values:")
negative = pd.DataFrame({"Feature": [clean_label(c) for c in numeric_cols], "Train count": [(train[c]<0).sum() for c in numeric_cols],
                         "Test count": [(test[c]<0).sum() for c in numeric_cols], "Train min": [train[c].min() for c in numeric_cols],
                         "Test min": [test[c].min() for c in numeric_cols], "Test max": [test[c].max() for c in numeric_cols]}, index=numeric_cols)
negative = negative[(negative["Train count"]>0) | (negative["Test count"]>0)]
display(negative)

The analysis above reveals potentially invalid values in `train` and `test` datasets. The features `prior_mean_purchase_value` and `spend_365d` contain extremely high values (1 billion), most likely due to data errors (placeholder). Additionally, negative values appear in time intervals and spending features, in both `train` and `test`, obviously impossible values. Negative values in `random_noise` are not necessarily problematic, as its name says, this variable can naturally take both positive and negative values. We will explore its distribution in some cells below.

### Cell 10 - Statistical analysis of numeric features:
Describe mean, standard deviation, quantiles, skew and count of zeros of each numeric feature in `train` dataset.

Then, each column is standardised using the z-score, so they can all be compared in a single boxplot figure. The resulting values are clipped to the interval [-4, 4] to allow only for a clearer visualisation, preserving the original values for further cleaning and preprocessing.

In [ ]:
desc = train[numeric_cols].describe().T
desc["skew"] = train[numeric_cols].skew()
desc["count of zeros"] = (train[numeric_cols] == 0).sum()
desc["zeros %"] = (train[numeric_cols] == 0).sum() / len(train) * 100
desc.insert(0, "label", [clean_label(c) for c in desc.index])
print(f"Numeric feature summary (train) - all {len(numeric_cols)} features:")
display(desc.round(2))

skewed = desc.index[desc["skew"].abs() > 1.5].tolist()
zero_heavy = desc.index[desc["zeros %"] > 60].tolist()
print("\nHighly skewed (|skew| > 1.5) -> log1p candidates in notebook 2:")
print("  " + (", ".join(clean_label(c) for c in skewed) if skewed else "none"))
print("\nZero-inflated (> 60% zeros) -> the 1.5*IQR boxplot rule is unreliable here:")
print("  " + (", ".join(clean_label(c) for c in zero_heavy) if zero_heavy else "none"))


mu = train[numeric_cols].mean()
sd = train[numeric_cols].std()

z_train = ((train[numeric_cols] - mu) / sd).clip(-4, 4)
z_long = pd.melt(z_train, var_name="feature", value_name="z_score")
z_long["label"] = [clean_label(f) for f in z_long["feature"]]
order = [clean_label(c) for c in desc["skew"].abs().sort_values(ascending=False).index.tolist()]
z_long = z_long.assign(_o=[order.index(l) for l in z_long["label"]]).sort_values("_o")

fig, ax = plt.subplots(figsize=(9, 11))
sns.boxplot(data=z_long, y="label", x="z_score", orient="h", color=PLOT_BLUE, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"All {len(numeric_cols)} numeric features, standardised on train (ordered by |skew|)")
ax.set_xlabel("z-score (train mean = 0, train sd = 1, clipped at ±4 for display)")
ax.set_ylabel("")
plt.setp(ax.get_yticklabels(), fontsize=8)
guide(ax, "How to read: one box per feature, in train sd; long right whisker = skew, box at 0 = zero-inflated.")
plt.savefig(FIG_DIR / "nb01_02_numeric_boxplots.png", dpi=200)
plt.tight_layout()
plt.show()

The statistical analysis above reveals that most numeric features are strongly right-skewed, particularly `Total Desc_ Global`, `prior_max_purchase_value`, `spend_365d`, `Total Portes` and `prior_mean_purchase_value` (skew > 18). For features with skew > 1.5, we listed candidates for log1p outlier cleaning method in notebook 2.

`Total Portes`, `Total Desc_ Global`, `purchase_count_30d` and `spend_30d` have more than 60% zero values, suggesting that the IQR method may be unreliable for identifying outliers in these features. Looking at the features mentioned earlier with outliers equal to 10^9, it's easy to conclude that these values significantly shift the mean and make the boxplots difficult to visualize. Overall, this analysis proposes that some features may benefit from transformations to reduce skewness, after addressing invalid values during preprocessing.

### Cell 11 - Histograms and outlier fences:
Show the shape of every numeric feature and quantify what the IQR rule would call outlier.

Values greater than or equal to 10^8 are excluded so a placeholder we discussed in cell 9 do not interfere with the display of the histograms.

In [ ]:
VALID_MAX = 1e8

def valid_values(col):
    s = train[col].dropna()
    return s[s < VALID_MAX]

print(f"All {len(numeric_cols)} histograms in one figure:")

N_COLS = 6
n_rows = math.ceil(len(numeric_cols) / N_COLS)
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(3.6 * N_COLS, 3.2 * n_rows))
for k, (ax, col) in enumerate(zip(axes.ravel(), numeric_cols)):
    s = valid_values(col)
    sns.histplot(s, bins=40, color=PLOT_BLUE, ax=ax)
    ax.axvline(s.median(), color=PLOT_ORANGE, linestyle="-", label="median")
    ax.axvline(s.mean(), color="black", linestyle="--", label="mean")
    ax.set_title(clean_label(col), fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("Count" if k % N_COLS == 0 else "")
    ax.tick_params(axis="x", labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
    if k == 0:
        ax.legend(loc="upper right")

last_ax = axes.flat[-1]

last_ax.clear()
last_ax.set_axis_off()
last_ax.grid(False)
last_ax.set_facecolor("white")

note = ("Reading guide:\n\n"
    " One panel per numeric feature, in dataset order.\n"
    " Orange line = median; black dashed line = mean.\n"
    " Values >= 10^8 are excluded.\n"
    " Bars are counts: a tall spike reflects many repeated values,\n"
    "not necessarily large values.\n"
    " The fence table below covers all features.")

last_ax.text(0.03, 0.97, note, transform=last_ax.transAxes, ha="left", va="top", fontsize=10, wrap=True,
    bbox=dict(facecolor="white", edgecolor="lightgray", boxstyle="round,pad=0.5"))
plt.savefig(FIG_DIR / "nb01_03_histograms_all.png", dpi=200)
plt.tight_layout()
plt.show()

fence_rows = []
for col in numeric_cols:
    s = valid_values(col)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    outside = int(((s < lo) | (s > hi)).sum())
    fence_rows.append({"label": clean_label(col), "feature": col, "q1": q1, "q3": q3,
                       "fence_low": lo, "fence_high": hi, "count of outliers": outside,
                       "outliers %": outside/len(s)*100})
fences = pd.DataFrame(fence_rows).set_index("label").sort_values("outliers %", ascending=False)
print("\nIQR outlier fences (1.5*IQR) on valid values (all numeric features, worst first):")
display(fences.round(2))

As expected, the histograms show strong right-skewed asymmetry, with long right tails and most observations concentrated at lower values. The exception is `random_noise`, which has a well-defined Gaussian curve. The IQR analysis identifies substantial proportions of potential outliers, particularly in `spend_30d` (16.39%) and `Total Portes` (16.28%). However, these values are related to the high number of zeros (cell 10) present in these features, resulting in short IQR limits, as can be seen in `Total Portes`, where any value other than 0 is considered an outlier. On the other hand, due to the high variability of values in `customer_tenure_days`, this feature does not exhibit any outliers.

In general, these results should be interpreted with caution, since many of the flagged observations may represent actual purchases rather than data errors.

### Cell 12 - Statistical analysis for categorical features:
Analyse each level of each categorical variable.

Potential placeholders may be present, identified in strings such as "unknown", "-" or "?".

In [ ]:
PLACEHOLDERS = {"?", "unknown", "n/a", "na", "none", "null", "nan", "-", ""}
for col in categorical_cols:
    vc = train[col].value_counts()
    print(f"\n{clean_label(col)} (raw: {col})")
    print(f"  levels: {train[col].nunique():3d} | rows with a value: {int(vc.sum())} | missing: {int(train[col].isna().sum())}")
    listing = pd.DataFrame({"level": vc.index.astype(str), "count": vc.values, "rows %": vc.values / len(train) * 100}).set_index("level")
    display(listing)
    ph = sorted({str(x).strip().lower() for x in vc.index if str(x).strip().lower() in PLACEHOLDERS})
    if ph:
        print("placeholder-like levels detected:", ph)

The categorical analysis reveals unbalanced distributions across several features, such as `commercial_zone` and `warehouse`, highly concentrated in a single level, accounting for approximately 96% of the observations. Similarly, `payment_type` and `salesperson` are dominated by a few levels, while `transport_method` has the highest cardinality (80 level), including many levels with only one observation. `document_series` exhibits a more balanced distribution across its three levels. There are missing values in all categorical features, but no placeholder-like levels were detected. Overall, these findings suggest that rare levels, highly concentrated features should be taken into account during preprocessing and feature selection.

### Cell 13 - Correlation between numeric features and the target:
Measure how much each numeric feature alone says about the target.

For each feature, it's used the point-biserial correlation coefficient (a Pearson between numeric and binary). The resultas are the correlation coefficient **r**, its absolute value, the p-value and the number of valid observations. Missing and invalid values are ommited so that the calculations are not distorted. A bar chart shows the correlation analysis, sorted in descending order by |r|.

In [ ]:
rows = []
for col in numeric_cols:
    if col == "random_noise":
        valid = train.loc[train[col] < 1e8, [col, TARGET]].dropna()
    else:
        valid = train.loc[(train[col] >= 0) & (train[col] < 1e8), [col, TARGET]].dropna()
    if valid[col].nunique()<2:
        continue
    pb = pointbiserialr(valid[TARGET], valid[col])
    rows.append({"feature": col, "r": pb.statistic, "|r|": abs(pb.statistic),
                 "p_value": pb.pvalue, "n_used": len(valid)})

assoc = pd.DataFrame(rows).sort_values("|r|", ascending=False).reset_index(drop=True)
assoc.insert(0, "label", [clean_label(f) for f in assoc["feature"]])
assoc = assoc.set_index("label")
print(f"Numeric association with target - all {len(assoc)} features (point-biserial r)")
display(assoc.round(4))

fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(data=assoc, y="label", x="r", orient="h", color=PLOT_BLUE, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Correlation with the target - ALL {len(assoc)} numeric features (sorted by |r|)")
ax.set_xlabel("Point-biserial r")
ax.set_ylabel("")
plt.setp(ax.get_yticklabels(), fontsize=8)
guide(ax, "How to read: bar = correlation with the target; right = more likely to repeat.")
plt.savefig(FIG_DIR / "nb01_04_association_numeric.png", dpi=200)
plt.tight_layout()
plt.show()

This analysis shows that historical purchasing features (counts and spends) have the strongest association with the target, mainly the purchases counts over the last 90, 180 and 365 days. In contrast, purchase gaps and `days_since_previous_purchase` show negative correlations, perhaps because customers with longer intervals between purchases are less likely to return within 90 days.

Features related to purchase amounts show weak correlations, while `random_noise`, `Total Desc_ Global` and `Total Portes` show non-significant linear associations. We can conclude that historical purchasing frequency appears to be more informative than individual transaction characteristics.

### Cell 14 - Correlation among numeric features:
Analyse linear relationships between numeric features using Pearson's correlation coefficient.

A correlation heatmap is generated to visualise the strength and direction of these relationships. We also look for pairs of features with an absolute coefficient of at least 0.80, which are presented in a separate table to highlight potential cases of redundancy, important for feature selection.

In [ ]:
corr = train[numeric_cols].corr(numeric_only=True)

CORR_ANNOT_THRESH = 0.30
annot_txt = corr.round(2).astype(str)
annot_txt = annot_txt.mask(corr.abs() < CORR_ANNOT_THRESH, "")
corr_lbl = corr.copy()
corr_lbl.index = [clean_label(c) for c in corr_lbl.index]
corr_lbl.columns = [clean_label(c) for c in corr_lbl.columns]
annot_lbl = annot_txt.copy()
annot_lbl.index = corr_lbl.index
annot_lbl.columns = corr_lbl.columns

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_lbl, annot=annot_lbl, fmt="", vmin=-1, vmax=1, cmap="RdYlGn", ax=ax)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=7)
ax.set_title(f"Pearson correlation among ALL numeric predictors (full matrix; blank = |r| < {CORR_ANNOT_THRESH:.2f})")
guide(ax, "How to read: green positive, red negative, fixed -1 to +1; blank cells are |r| below 0.30, not missing.")
plt.savefig(FIG_DIR / "nb01_05_correlation_heatmap.png", dpi=200)
plt.tight_layout()
plt.show()

cols = corr.columns.tolist()
pairs = []
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        v = corr.loc[a, b]
        if pd.notna(v) and abs(v) >= 0.80:
            pairs.append({"feature_1": a, "feature_2": b, "pearson_r": v})
pairs_df = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "pearson_r"])
if len(pairs_df):
    pairs_df = pairs_df.sort_values("pearson_r", key=abs, ascending=False)
    pairs_df.insert(0, "label_1", [clean_label(c) for c in pairs_df["feature_1"]])
    pairs_df.insert(1, "label_2", [clean_label(c) for c in pairs_df["feature_2"]])
print(f"\nPredictor pairs with |Pearson r| >= 0.80: {len(pairs_df)}")
if len(pairs_df):
    display(pairs_df.style.hide(axis="index"))
else:
    print("none")

The correlation heatmap shows 2 big groups of strongly positively correlated features: purchase amounts and prior purchase counts and spends. Regarding the most negative correlations (r ≈ -0.3), the focus shifts to purchase gaps and purchase frequency. As mentioned earlier, this suggests that customers with longer intervals between purchases tend to purchase less frequently.

We identified 16 pairs of numeric features with strong linear relationships (|r| ≥ 0.8). The strongest correlations are found among features related to purchase amounts, particularly `Total a Pagar`, `Total Liquido` and `Total Bruto` (r > 0.97). Purchase counts and spends over time periods also show strong positive correlations, especially between `purchase_count_180d` and `purchase_count_365d` (r = 0.95), which is expected given their overlapping periods. This exploration should be considered during feature selection, in notebook 3.

### Cell 15 - Correlation between categorical features and the target:
Measure association for the categorical features and show where the repeat purchase rate (`repeat_purchase_90d`=1) actually moves.

The features are ranked by their association strength using Cramér's V from chi-squared contigency tables. Bar charts are used to compare the repeat purchase rates across their different categories. Each bar includes the number of observations in that category and the dashed line represents the overall repeat purchase rate, allowing us to identify categories with above higher rate than that baseline. For readability, only a maximum of 12 levels per feature are displayed.

In [ ]:
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    return association(table.values, method="cramer", correction=True)


cat_rows = []
for col in categorical_cols:
    valid = train[[col, TARGET]].dropna()
    cat_rows.append({"feature": col, "levels": int(train[col].nunique()),
                     "Cramér's V": cramers_v(valid[col], valid[TARGET])})
cat_df = pd.DataFrame(cat_rows).sort_values("Cramér's V", ascending=False)
cat_df.insert(0, "label", [clean_label(c) for c in cat_df["feature"]])
cat_df = cat_df.set_index("label")
print(f"Categorical association with target (Cramér's V) - all {len(cat_df)} features")
display(cat_df)

base = train[TARGET].mean() * 100
PANEL_LEVEL_CAP = 12
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for k, (ax, col) in enumerate(zip(axes.ravel(), categorical_cols)):
    rate = train.groupby(col)[TARGET].mean() * 100
    nsup = train[col].value_counts()
    tbl = pd.DataFrame({"n": nsup, "rate %": rate}).dropna()
    capped = len(tbl) > PANEL_LEVEL_CAP
    if capped:
        tbl = tbl.sort_values("n", ascending=False).head(PANEL_LEVEL_CAP)
    tbl = tbl.sort_values("rate %", ascending=False)
    sns.barplot(x=tbl["rate %"].values, y=tbl.index.astype(str), orient="h", color=PLOT_BLUE, ax=ax)
    for i, (r, n) in enumerate(zip(tbl["rate %"].values, tbl["n"].values)):
        ax.text(r - 1, i, f"n={int(n)}", ha="right", fontsize=9, color="white")
    ax.axvline(base, color=PLOT_ORANGE, linestyle="--")
    ax.set_title("{}{} | baseline {:.1f}%".format(
        clean_label(col),
        " ({} most frequent levels)".format(PANEL_LEVEL_CAP) if capped else "", base))
    ax.set_xlabel("Repeat purchase rate (%)")
    ax.set_ylabel("")
    plt.setp(ax.get_yticklabels(), fontsize=7)
    if capped:
        print(f"{clean_label(col)} has {train[col].nunique()} levels: the {PANEL_LEVEL_CAP} most frequent are plotted, all {train[col].nunique()} are listed in Cell 12")
guide(axes.ravel()[0], "How to read: bar = repeat rate per level, n = rows behind it, dashed = baseline.")
plt.savefig(FIG_DIR / "nb01_06_categorical_rates.png", dpi=200)
plt.tight_layout()
plt.show()

The categorical correlation analysis shows that `transport_method` has the strongest association with the target (Cramér's V = 0.290), followed by `payment_type` (0.238) and `salesperson` (0.203). However, there is a gap between these three features and the rest: `commercial_zone`, `warehouse` and `document_series` have almost zero correlation. The bar charts also reveal only 4 out of 6 features had levels that significantly exceeded the baseline (`payment_type`, `salesperson`, `transport_method`, `warehouse`). These differences should be interpreted carefully, particularly for categories with few observations. For example, in the `salesperson` feature, all customers served by VLOOKUP Master 8 repeated purchases, but there were only 3 observations.

### Cell 16 - Customer metadata:
Explore if customers features in `clientes` dataset could provide additional information for predicting repeat purchases.

The dataset structure and feature cardinality are examined, along with the overlap of customers between `train` and `test` datasets. `clientes` dataset is merged to the `train` data through `customer_id` and repeat purchase rates are compared across all customer features. To ensure more reliable analysis, only features with at most 40 distinct levels and containing at least 100 observations are considered. The results are presented in tables and bar charts with the overall repeat purchase rate as previously mentioned used as a baseline, to evaluate the potential relevance of these features for the model.

In [ ]:
print("Clientes dataset profile:")
print(" rows / unique customer_id:", len(clientes), "/", clientes["customer_id"].nunique())
print(" registration year range  :", int(clientes["customer_since_year"].min()), "-", int(clientes["customer_since_year"].max()))
print("\nAttribute cardinality (nunique):")
card = clientes.nunique().to_frame("n_unique")
card.insert(0, "label", [clean_label(c) for c in card.index])
display(card)

train_cov = train["customer_id"].isin(clientes["customer_id"]).sum() / len(train) * 100
test_cov = test["customer_id"].isin(clientes["customer_id"]).sum() / len(test) * 100
overlap = train.loc[train["customer_id"].isin(test["customer_id"]), "customer_id"].nunique()
print("\nJoin coverage on customer_id:")
print(f" train rows matched in clientes  : {train_cov:.1f}%")
print(f" test rows matched in clientes   : {test_cov:.1f}%")
print(" customers in BOTH train and test:", overlap)

train_merged = train.set_index("customer_id").join(clientes.set_index("customer_id")).reset_index()
base = train[TARGET].mean() * 100
MIN_GROUP = 100 
MAX_LEVELS = 40


def rate_table(col):
    rate = train_merged.groupby(col)[TARGET].mean() * 100
    tbl = pd.DataFrame({"n": train_merged[col].value_counts(), "rate %": rate})
    tbl["dev_pp"] = (tbl["rate %"] - base).abs()
    return tbl[tbl["n"] >= MIN_GROUP]


meta_cols = [c for c in clientes.columns
             if c != "customer_id" and clientes[c].notna().any()
             and clientes[c].nunique() <= MAX_LEVELS]
skipped = [c for c in clientes.columns if c != "customer_id" and c not in meta_cols]
print(f"\nAttributes analysed (rule: at least one value present and <= {MAX_LEVELS} distinct levels):")
print(" analysed :", [clean_label(c) for c in meta_cols])
print(" excluded :", [clean_label(c) for c in skipped])
print(" -> excluded columns are free text / near-unique identifiers (city, street, address, postal_code, company_name, ...) or fully empty (customer_type)")
print("they are not model inputs, and notebook 2 treats them as identifiers, not as features.")

tables = {}
for col in meta_cols:
    tbl = rate_table(col)
    if not len(tbl):
        print(f"\n {clean_label(col)} [raw: {col}] === no level reaches n >= {MIN_GROUP}")
        continue
    tables[col] = tbl
    print(f"\n{clean_label(col)} [raw: {col}]:")
    print(f" {len(tbl)} of {int(train_merged[col].nunique())} levels have n >= {MIN_GROUP}; those levels cover {tbl['n'].sum() / len(train_merged) * 100:.1f}% of joined rows")
    display(tbl.sort_values("rate %", ascending=False).round(1))

show_cols = list(tables)
n_panels = len(show_cols)
n_cols = 2 if n_panels > 2 else max(n_panels, 1)
n_rows = -(-n_panels // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 4.2 * n_rows))
axes = list(axes.ravel()) if n_panels > 1 else [axes]
for ax, col in zip(axes, show_cols):
    tbl = tables[col].sort_values("rate %", ascending=False)
    sns.barplot(x=tbl["rate %"].values, y=tbl.index.astype(str), orient="h", color=PLOT_BLUE, ax=ax)
    ax.axvline(base, color=PLOT_ORANGE, linestyle="--")
    ax.set_title(f"{clean_label(col)} (n >= {MIN_GROUP}) | baseline {base:.1f}%")
    ax.set_xlabel("Repeat rate (%)")
    ax.set_ylabel("")
    plt.setp(ax.get_yticklabels(), fontsize=7)
guide(axes[0], "How to read: bar = repeat rate per level (n >= 100); dashed line = baseline.")
plt.savefig(FIG_DIR / "nb01_07_clientes_attributes.png", dpi=200)
plt.tight_layout()
plt.show()

for col, tbl in tables.items():
    top_lvl = tbl["dev_pp"].idxmax()
    print(f"  {clean_label(col):28s}: most deviating trusted level = {top_lvl} | n = {int(tbl.loc[top_lvl, 'n'])} | rate = {tbl.loc[top_lvl, 'rate %']:.1f}% | dev = {tbl.loc[top_lvl, 'dev_pp']:.1f} pp")

The analysis of the `clientes` dataset confirms that each of the 1143 customers has a unique identifier with 100% coverage in both `train` and `test`, 184 customers appear in both datasets.

Repeat purchase rates show considerable differences across customer features when compared to the baseline. As we have seen, `commercial_region` attribute shows the greatest variation, with rates ranging from 44.2% to 94.5% in sufficiently represented features. `customer_since_year` (25.2% to 66.8%) and `island` (32.7% to 62.7%) also show wide variations. This suggests that customer metadata can provide useful predictive information for the model, supported by the integration of specific features during preprocessing and feature selection.

### Cell 17 - Train versus test:
Identify differences between `train` and `test` numeric feature distributions (**data drift**).

For each feature, the means are compared using percentage differences and differences standardised by the training standard deviation. `purchase_date` ranges are also examined to check their temporal coverage. We visualize the distributions using standardised boxplots (`train` z-scores, clipped to [-4, 4] as done previously). Histograms are presented for the 2 features with the largest standardised mean differences, providing a more detailed comparison of their distributions.

In [ ]:
shift = pd.DataFrame({"train_mean": train[numeric_cols].mean(), "test_mean": test[numeric_cols].mean(), "train_std": train[numeric_cols].std()})
shift["mean_diff %"] = (shift["test_mean"] - shift["train_mean"]) / shift["train_mean"].abs().replace(0, np.nan) * 100
shift["std_diff"] = (shift["test_mean"] - shift["train_mean"]) / shift["train_std"].replace(0, np.nan)
shift["abs_std_diff"] = shift["std_diff"].abs()
shift = shift.sort_values("abs_std_diff", ascending=False)
shift.insert(0, "label", [clean_label(c) for c in shift.index])
print("Train -> test shift for all numeric features (sorted by |std_diff|):")
display(shift.set_index("label").style.format("{:.2f}"))

print("\nPurchase_date coverage:")
print(" train:", train["purchase_date"].dropna().min(), "->", train["purchase_date"].dropna().max())
print(" test :", test["purchase_date"].dropna().min(), "->", test["purchase_date"].dropna().max())


zt = ((train[numeric_cols] - mu) / sd).clip(-4, 4)
ze = ((test[numeric_cols] - mu) / sd).clip(-4, 4)
both = pd.concat([zt.assign(split="train"), ze.assign(split="test")])
both_long = pd.melt(both, id_vars="split", var_name="feature", value_name="z_score")
both_long["label"] = [clean_label(f) for f in both_long["feature"]]
fig, ax = plt.subplots(figsize=(9, 11))
sns.boxplot(data=both_long, y="label", x="z_score", hue="split", orient="h", palette=[PLOT_BLUE, PLOT_ORANGE], ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Train vs test, all numeric features (standardised on train)")
ax.set_xlabel("z-score (train mean = 0, train sd = 1, clipped at +/-4 for display)")
ax.set_ylabel("")
plt.setp(ax.get_yticklabels(), fontsize=8)
ax.legend()
guide(ax, "How to read: blue = train, orange = test, both in train sd; a shifted orange box = drift.")
plt.savefig(FIG_DIR / "nb01_08_drift_boxplots.png", dpi=200)
plt.tight_layout()
plt.show()

drift_cols = shift.index[:2].tolist()
print("\nOverlaid histograms for the two largest |std_diff|:", [clean_label(c) for c in drift_cols])
print(f"Note: test holds ~{len(train) / len(test):.1f}x fewer rows than train, so compare distribution shapes, not heights.")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col in zip(axes, drift_cols):
    sns.histplot(train[col].dropna(), bins=40, color=PLOT_BLUE, label="train", ax=ax)
    sns.histplot(test[col].dropna(), bins=40, color=PLOT_ORANGE, label="test", ax=ax)
    ax.set_title("Train vs test: " + clean_label(col))
    ax.set_xlabel("")
    ax.legend()
guide(axes[0], "How to read: blue = train, orange = test; compare SHAPES, not heights (test has fewer rows).")
plt.savefig(FIG_DIR / "nb01_09_drift_histograms.png", dpi=200)
plt.tight_layout()
plt.show()

The comparison above shows the highest shifts occur in `customer_tenure_days` (+120.33% in mean, sd=1.45), `prior_total_spend` (+161.43% in mean, sd=0.98) and `Total Desc_ Linha` (+132.4% in mean, sd=0.58).

The standardised boxplots confirm these shifts, with the `test` dataset generally showing higher medians and wider IQR ranges, mainly for features related to purchase amounts and spends in time periods. Features such as `random_noise`, `Total Desc_ Global` and prior purchase gaps features show similar distributions across both datasets. Many monetary features, such as `Total a Pagar`. have numerous extremely high observations in both datasets, reflecting their strong right-skewness.

Additionally, the purchase dates confirm that `test` contains more recent observations (2025-2026) than `train` (2015-2025), which may explain the observed differences. Through this analysis, we notice that there may be data drift, with a particular focus on customer history and spending features, and that should considered when evaluating the model's performance to predict future purchases.

### Cell 18 - Summary:

In [ ]:
n_missing_cols = int((train.isna().sum() > 0).sum())
print("Data Exploration summary:")
print(f"1. Size: train={len(train)} rows, test={len(test)} rows; {train['customer_id'].nunique()} unique customers.")
print(f"2. Balance: {share.loc[1]*100:.1f}% positive / {share.loc[0]*100:.1f}% negative (minority-majority ratio { balance_ratio:.2f}) -> close to  balanced; majority baseline {baseline*100:.1f}%.")
print(f"3. Missing: {n_missing_cols} columns contain nulls; {int((row_missing == row_missing.max()).sum())} train rows are structurally empty.")
print(f"4. Invalid values: 10^9 placeholders and impossible negatives.")
print(f"5. Strongest numeric associations: {', '.join(clean_label(f) for f in assoc.head(3)['feature'])}.")
_cramer_s_v = "Cramér's V"
print(f"6. Strongest categorical associations: {clean_label(cat_df['feature'].tolist()[0])} (Cramér's V={cat_df[_cramer_s_v].tolist()[0]:.3f}).")
print(f"7. Redundant predictor pairs (|r| >= 0.80): {len(pairs_df)}.")
best_col, best_dev = max(((c, t["dev_pp"].max()) for c, t in tables.items()), key=lambda t: t[1])
print(f"8. Clients dataset carries real association: largest trusted deviation {best_dev:.1f} pp in {clean_label(best_col)}.")
print("9. Train vs test drift exists and points to temporal/grouped validation.")
print(f"10. {clean_label('random_noise')} is a planted noise column (near-zero association).")


print("---------")
train.to_csv(DATA_DIR / "train_raw.csv", index=False)
test.to_csv(DATA_DIR / "test_raw.csv", index=False)
clientes.to_csv(DATA_DIR / "clientes_raw.csv", index=False)
print("Saved to", DATA_DIR)
print("  train_raw.csv, test_raw.csv, clientes_raw.csv")

---

## Where this notebook leaves the homework

- `data/train_raw.csv`, `data/test_raw.csv`, `data/clientes_raw.csv`: unchanged raw datasets
-  9 figures in `artifacts/figures/`

**Conclusions Notebook 2 must act on:**

| Notebook 1 finding | Action taken in Notebook 2 |
|---|---|
| 65 train rows have every transaction field null; test has none |Dropped from `train`|
| 1e9 placeholders in Prior Mean Purchase Value and Spend (365d) | Converted to NaN + invalid-value flag. Never clipped into a real number |
| Impossible negatives in Spend (365d), Days Since Previous Purchase, Prior Mean Purchase Gap| Converted to NaN + negative-value flag |
| `prior...` nulls are structural: 603 first-time buyers and statistics features related to that are undefined after a single purchase | Filled 0 **only for those rows**, with `no_purchase_history` / `single_purchase` flags|
|Categorical nulls (~4% of rows) carry no meaning | Imputed an explicit `Missing` level, which one-hot encodes into its own column |
| `transport_method` has 80 levels, 59 of them below 1% of rows|Levels below 1% of train grouped into `Other` before encoding |
| `clientes` attributes with <= 40 levels carry real association with | Joined on `customer_id` and 100% null features dropped|
|`train` `purchase_date` between 2015-2025, `test` 2025-2026 | `purchase_year` computed only for data quality assurance but **excluded from the model** (a coefficient on a year cannot extrapolate past the training range); `purchase_month` is kept for the model |
| 29 features with \|skew\| > 1.5 and high zero-inflation | `log1p` on the skewed non-negative columns before scaling |
|`random_noise` is a planted column with near-zero association|Dropped|




**Next: Notebook 2 - Clean & Preprocess**